In [1]:
import os, glob
import numpy as np
import pandas as pd
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
import cv2
import re
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch

2026-01-15 22:56:50.043044: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-15 22:56:50.055636: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768514210.068473 2678575 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768514210.072359 2678575 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-15 22:56:50.087306: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
def gaussian_heatmap(h, w, cx, cy, sigma):
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    d2 = (xx - cx)**2 + (yy - cy)**2
    heat = np.exp(-d2 / (2 * (sigma**2)))
    return heat.astype(np.float32)

In [3]:
def split_by_scan(
    df_long,
    val_frac=0.2,
    test_frac=0.15,
    seed=42,
    path_col="full_path",
):

    assert 0.0 <= val_frac < 1.0, "val_frac must be in [0,1)"
    assert 0.0 <= test_frac < 1.0, "test_frac must be in [0,1)"
    assert val_frac + test_frac < 1.0, "val_frac + test_frac must be < 1"

    rng = np.random.default_rng(seed)
    scans = df_long[path_col].dropna().unique()
    rng.shuffle(scans)

    n_total = len(scans)
    n_test = int(round(n_total * test_frac))
    n_val  = int(round(n_total * val_frac))

    test_scans = set(scans[:n_test])
    val_scans  = set(scans[n_test:n_test + n_val])
    train_scans = set(scans[n_test + n_val:])

    df_train = df_long[df_long[path_col].isin(train_scans)].reset_index(drop=True)
    df_val   = df_long[df_long[path_col].isin(val_scans)].reset_index(drop=True)
    df_test  = df_long[df_long[path_col].isin(test_scans)].reset_index(drop=True)

    return df_train, df_val, df_test


In [4]:
def plot_images_with_landmarks(image_classes, df_coords):
    plt.figure(figsize=(12, 12))

    for i, category in enumerate(image_classes):
        image_path = destination_dir / category
        image_in_folder = os.listdir(image_path)
        
        # Vezmeme první obrázek dané třídy
        first_image = image_in_folder[0]
        first_image_path = image_path / first_image
        
        # Načteme obrázek
        img = image.load_img(first_image_path, target_size=(IMG_SIZE, IMG_SIZE))
        img_array = image.img_to_array(img) / IMG_SIZE

        # Najdeme anotaci pro tento soubor
        matched_row = df_coords[df_coords['full_path'].str.endswith(first_image)].iloc[0]
        x_px = int(matched_row['relative_x'] * IMG_SIZE)
        y_px = int(matched_row['relative_y'] * IMG_SIZE)

        # Vykreslení
        plt.subplot(4, 4, i + 1)
        plt.imshow(img_array)
        plt.scatter(x_px, y_px, c='red', s=40)
        plt.title(f"{category}\n{first_image} ({matched_row['level']})")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [5]:
def generate_heatmap(x, y, sigma):
    xx, yy = np.meshgrid(np.arange(IMG_SIZE), np.arange(IMG_SIZE))
    heatmap = np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * sigma**2))
    return heatmap

In [6]:
def compute_mean_std(dataset, batch_size=16):
    loader = DataLoader(dataset, batch_size=batch_size,
                        shuffle=False, num_workers=4)

    n_pixels = 0
    ch_sum = 0
    ch_sum_sq = 0

    for imgs, _ in tqdm(loader, desc="Computing mean/std"):
        imgs = imgs.float()     # [B, C, H, W]
        B, C, H, W = imgs.shape
        imgs = imgs.view(B, C, -1)

        ch_sum    += imgs.sum(dim=[0, 2])
        ch_sum_sq += (imgs ** 2).sum(dim=[0, 2])
        n_pixels  += B * H * W

    mean = ch_sum / n_pixels
    std  = torch.sqrt(ch_sum_sq / n_pixels - mean ** 2)
    return mean, std

In [7]:
# Formátování inputu pro síť 
def canonical_level_name(level: str) -> str:
    """Normalize level text for channel naming."""
    lvl = str(level).strip()
    lvl = lvl.replace("/", "_")      # L5/S1 -> L5_S1
    lvl = re.sub(r"\s+", "", lvl)    # remove spaces
    return lvl


In [8]:
def train_one_epoch(model, loader, optimizer, device, criterion):
    model.train()
    total_loss = 0

    for imgs, targets in loader:
        imgs    = imgs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        # forward under autocast
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            outputs = model(imgs)
            loss = criterion(outputs, targets)

        # backward under GradScaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)

    return total_loss / len(loader.dataset)



In [9]:
@torch.no_grad()
def eval_one_epoch(model, loader, device, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for imgs, targets in loader:
            imgs    = imgs.to(device)
            targets = targets.to(device)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                outputs = model(imgs)
                loss = criterion(outputs, targets)

            total_loss += loss.item() * imgs.size(0)

    return total_loss / len(loader.dataset)


In [10]:
@torch.no_grad()
def evaluate_model(model, dataset, device):
    model.eval()

    K = len(dataset.dataset.levels)  # because val_ds is a Subset
    per_level_errors = [[] for _ in range(K)]

    for img_t, gt_t in tqdm(dataset, desc="Testing"):
        img = img_t.unsqueeze(0).to(device)    # [1,3,H,W]
        pred = model(img)[0].cpu()             # [K,H,W]

        for k in range(K):
            pred_hm = pred[k]
            gt_hm   = gt_t[k]

            err = localization_error(pred_hm, gt_hm)
            per_level_errors[k].append(err)

    return per_level_errors

In [11]:

import torch
import numpy as np

def denormalize(img_t, mean, std):
    """
    Grayscale-only denormalization.

    img_t: [1,H,W] normalized tensor
    mean, std: scalar, list of length 1, or 1-element tensor

    Returns:
        [H,W] numpy array in [0,1]
    """
    img = img_t.clone().cpu()

    # ensure mean/std are 1D tensors of length 1
    if not torch.is_tensor(mean):
        mean = torch.tensor(mean, dtype=img.dtype)
    if not torch.is_tensor(std):
        std = torch.tensor(std, dtype=img.dtype)

    mean = mean.view(1, 1, 1)   # [1,1,1]
    std  = std.view(1, 1, 1)    # [1,1,1]

    img = img * std + mean
    img = img.clamp(0.0, 1.0)

    # [1,H,W] -> [H,W]
    img = img.squeeze(0).numpy()
    return img


In [12]:
def heatmap_argmax(hm):
    """
    hm: [H, W] heatmap (torch tensor or numpy)
    returns (x, y) coordinate of the peak
    """
    flat_index = hm.argmax()
    H, W = hm.shape
    y = (flat_index // W).item()
    x = (flat_index %  W).item()
    return x, y

In [13]:
def localization_error(pred_hm, gt_hm):
    """
    pred_hm: [H,W]
    gt_hm:   [H,W]
    Returns pixel distance between predicted and GT peak.
    """
    px, py = heatmap_argmax(pred_hm)
    gx, gy = heatmap_argmax(gt_hm)
    return math.sqrt((px - gx)**2 + (py - gy)**2)

In [14]:
@torch.no_grad()
def visualize_sample(model, dataset, idx, device, mean, std, level_idx=0,
                     show_points=True):
    """
    model: trained UNet
    dataset: e.g. val_ds (Subset or full dataset)
    idx: index within that dataset
    level_idx: which channel (vertebra) to show

    Assumes:
      - grayscale input: img_t shape [1,H,W]
      - denormalize(...) returns [H,W] in [0,1]
    """
    model.eval()

    # img_t: [1,H,W], target_t: [K,H,W]
    img_t, target_t = dataset[idx]

    # [H,W] numpy in [0,1]
    img_vis = denormalize(img_t, mean, std)

    gt_hm = target_t[level_idx].cpu()          # [H,W]
    img_b = img_t.unsqueeze(0).to(device)      # [1,1,H,W]
    pred  = model(img_b)[0, level_idx].cpu()   # [H,W]

    # normalize heatmaps for display
    gt_hm_np   = gt_hm.numpy()
    pred_hm_np = pred.numpy()
    gt_disp    = gt_hm_np / (gt_hm_np.max()   + 1e-8)
    pred_disp  = pred_hm_np / (pred_hm_np.max() + 1e-8)

    # optional landmark points (argmax of GT and prediction)
    if show_points:
        gx, gy = heatmap_argmax(gt_hm)
        px, py = heatmap_argmax(pred)

    plt.figure(figsize=(12, 4))

    # 1) raw MRI
    plt.subplot(1, 3, 1)
    plt.title("MRI")
    plt.imshow(img_vis, cmap="gray")
    plt.axis("off")

    # 2) GT heatmap overlay
    plt.subplot(1, 3, 2)
    plt.title(f"GT heatmap (level {level_idx})")
    plt.imshow(img_vis, cmap="gray", alpha=0.7)
    plt.imshow(gt_disp, cmap="hot", alpha=0.5)
    if show_points:
        plt.scatter([gx], [gy], c="lime", s=40, marker="x")
    plt.axis("off")

    # 3) Pred heatmap overlay
    plt.subplot(1, 3, 3)
    plt.title(f"Pred heatmap (level {level_idx})")
    plt.imshow(img_vis, cmap="gray", alpha=0.7)
    plt.imshow(pred_disp, cmap="hot", alpha=0.5)
    if show_points:
        plt.scatter([px], [py], c="cyan", s=40, marker="x")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


In [15]:
def heatmap_confidence(hm: torch.Tensor, radius: int = 15):
    """
    Returns:
      confidence: scalar (higher = more confident)
      peaks: ((m1,(x1,y1)), (m2,(x2,y2)))
    """
    (m1, p1), (m2, p2) = find_two_peaks(hm, radius=radius)

    if (m2 is None) or (m2 <= 0):
        return float("inf"), ((m1, p1), (m2, p2))

    conf = m1 / (m2 + 1e-8)
    return conf, ((m1, p1), (m2, p2))

In [16]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def hm_argmax_xy(hm_2d: torch.Tensor):
    # hm_2d: [H,W]
    idx = torch.argmax(hm_2d).item()
    H, W = hm_2d.shape
    y = idx // W
    x = idx % W
    return x, y

def extract_points_from_heatmaps(hm: torch.Tensor):
    # hm: [K,H,W]
    pts = []
    for k in range(hm.shape[0]):
        x, y = hm_argmax_xy(hm[k])
        pts.append((x, y))
    return pts

def show_sample_pair(unaug_ds, aug_ds, idx, epoch=0, vmax_hm=None):
    # Set curriculum epoch for augmented dataset
    if hasattr(aug_ds, "set_epoch"):
        aug_ds.set_epoch(epoch)

    x0, y0 = unaug_ds[idx]     # x0: [1,H,W] normalized, y0: [K,H,W]
    x1, y1 = aug_ds[idx]

    # de-normalize for display (assumes dataset normalized with mean/std)
    def denorm(x, mean, std):
        return (x * (std + 1e-8) + mean)

    mean = getattr(unaug_ds, "mean", 0.0)
    std  = getattr(unaug_ds, "std", 1.0)

    img0 = denorm(x0, mean, std).squeeze(0).numpy()
    img1 = denorm(x1, mean, std).squeeze(0).numpy()

    pts0 = extract_points_from_heatmaps(y0)
    pts1 = extract_points_from_heatmaps(y1)

    K = y0.shape[0]
    hm0_sum = y0.sum(dim=0).numpy()
    hm1_sum = y1.sum(dim=0).numpy()

    if vmax_hm is None:
        vmax_hm = max(hm0_sum.max(), hm1_sum.max(), 1e-6)

    fig, axs = plt.subplots(2, 2, figsize=(10, 10))
    axs[0,0].imshow(img0, cmap="gray")
    axs[0,0].set_title("Original image")
    axs[0,0].axis("off")

    axs[0,1].imshow(img1, cmap="gray")
    axs[0,1].set_title(f"Augmented image (epoch={epoch})")
    axs[0,1].axis("off")

    axs[1,0].imshow(img0, cmap="gray")
    axs[1,0].imshow(hm0_sum, cmap="magma", alpha=0.45, vmin=0, vmax=vmax_hm)
    axs[1,0].set_title("Original + sum(heatmaps)")
    axs[1,0].axis("off")

    axs[1,1].imshow(img1, cmap="gray")
    axs[1,1].imshow(hm1_sum, cmap="magma", alpha=0.45, vmin=0, vmax=vmax_hm)
    axs[1,1].set_title("Augmented + sum(heatmaps)")
    axs[1,1].axis("off")

    # plot keypoints
    for k, (x,y) in enumerate(pts0):
        axs[0,0].scatter(x, y, s=25)
        axs[1,0].scatter(x, y, s=25)
    for k, (x,y) in enumerate(pts1):
        axs[0,1].scatter(x, y, s=25)
        axs[1,1].scatter(x, y, s=25)

    plt.tight_layout()
    plt.show()

    # Optional: print curriculum params for this epoch
    if hasattr(aug_ds, "current_params") and aug_ds.current_params is not None:
        print("Curriculum params:", aug_ds.current_params)


In [17]:
@torch.no_grad()
def eval_one_epoch(model, loader, device, criterion, use_amp=False):
    model.eval()
    running = 0.0
    n = 0

    for imgs, targets in loader:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(imgs)
            loss = criterion(outputs, targets)

        bs = imgs.size(0)
        running += loss.item() * bs
        n += bs

    return running / max(1, n)

In [18]:
def train_one_epoch(model, loader, optimizer, device, criterion, scaler=None, use_amp=False):
    model.train()
    running = 0.0
    n = 0

    for imgs, targets in loader:
        imgs = imgs.to(device, non_blocking=True)           # [B,1,H,W]
        targets = targets.to(device, non_blocking=True)     # [B,K,H,W]

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(imgs)                           # [B,K,H,W]
            loss = criterion(outputs, targets)

        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        bs = imgs.size(0)
        running += loss.item() * bs
        n += bs

    return running / max(1, n)


In [19]:
def unwrap_dataset(ds):
    # If ds is a Subset, it has .dataset; otherwise return ds directly
    return ds.dataset if hasattr(ds, "dataset") else ds

from tqdm import tqdm
import numpy as np
import torch

@torch.no_grad()
def evaluate_model(model, data, device):
    """
    data can be:
      - a Dataset (e.g., SpineLandmarksDatasetAug)
      - a Subset
      - a DataLoader
    """
    model.eval()

    # If a DataLoader is passed, it has .dataset
    base_ds = unwrap_dataset(data.dataset) if hasattr(data, "dataset") else unwrap_dataset(data)
    K = len(base_ds.levels)

    per_level_errors = [[] for _ in range(K)]

    iterator = data
    desc = "Testing"
    if not hasattr(data, "__iter__") or isinstance(data, torch.utils.data.Dataset):
        # It's a Dataset/Subset; wrap it into a simple iterator
        iterator = data

    for batch in tqdm(iterator, desc=desc):
        # batch could be (img_t, gt_t) or batched (imgs, gts)
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            imgs, gts = batch
        else:
            raise ValueError("Expected (imgs, gts) from dataset/dataloader")

        # If coming from a Dataset: imgs shape is [1,H,W] or [C,H,W]
        # If coming from a DataLoader: imgs shape is [B,1,H,W]
        if imgs.dim() == 3:
            imgs = imgs.unsqueeze(0)  # -> [1,C,H,W]
            gts = gts.unsqueeze(0)    # -> [1,K,H,W]

        imgs = imgs.to(device)

        preds = model(imgs).cpu()     # [B,K,H,W]

        B = preds.shape[0]
        for b in range(B):
            for k in range(K):
                pred_hm = preds[b, k]
                gt_hm   = gts[b, k]   # keep on CPU fine

                err = localization_error(pred_hm, gt_hm)
                per_level_errors[k].append(float(err))

    return per_level_errors

def argmax_xy(hm):
    # hm: [H,W] tensor
    idx = torch.argmax(hm).item()
    H, W = hm.shape
    y = idx // W
    x = idx % W
    return x, y

def localization_error_argmax(pred_hm, gt_hm):
    px, py = argmax_xy(pred_hm)
    gx, gy = argmax_xy(gt_hm)
    return math.sqrt((px - gx)**2 + (py - gy)**2)


In [20]:
# =========================
# 9) Evaluation helpers: landmark error from heatmaps
# =========================
@torch.no_grad()
def heatmap_argmax_xy(hm):
    """
    hm: [B,K,H,W] -> (x,y) pixel coords [B,K,2]
    """
    B, K, H, W = hm.shape
    flat = hm.view(B, K, -1)
    idx = flat.argmax(dim=-1)  # [B,K]
    y = (idx // W).float()
    x = (idx %  W).float()
    return torch.stack([x, y], dim=-1)  # [B,K,2]


@torch.no_grad()
def mean_pixel_error(pred, target):
    """
    pred/target: [B,K,H,W]
    returns: scalar mean euclidean pixel error over B*K
    """
    pxy = heatmap_argmax_xy(pred)
    txy = heatmap_argmax_xy(target)
    err = torch.norm(pxy - txy, dim=-1)  # [B,K]
    return err.mean().item()

In [21]:

def _denorm_1ch(img_t, mean, std):
    """
    img_t: [1,H,W] torch tensor normalized
    returns: [H,W] numpy in [0,1] (clipped)
    """
    if isinstance(mean, (list, tuple)):
        mean = mean[0]
    if isinstance(std, (list, tuple)):
        std = std[0]
    img = img_t.detach().cpu().float()
    img = img * std + mean
    img = img.squeeze(0).numpy()
    img = np.clip(img, 0.0, 1.0)
    return img
    
def visualize_training_batch(
    loader,
    mean=[0.2127],
    std=[0.2673],
    level_idx=0,
    max_images=6,
    overlay_alpha=0.45,
    show_points=True,
    point_size=35
):
    """
    Shows up to max_images from the next batch in loader.
    Overlays heatmap channel = level_idx.
    """
    imgs, targets = next(iter(loader))  # imgs: [B,1,H,W], targets: [B,K,H,W]
    B, _, H, W = imgs.shape
    K = targets.shape[1]

    level_idx = int(np.clip(level_idx, 0, K - 1))

    n_show = min(B, max_images)
    n_cols = min(3, n_show)
    n_rows = int(np.ceil(n_show / n_cols))

    plt.figure(figsize=(4.2 * n_cols, 4.2 * n_rows))

    for i in range(n_show):
        img = _denorm_1ch(imgs[i], mean, std)
        hm = targets[i, level_idx].detach().cpu().float().numpy()

        ax = plt.subplot(n_rows, n_cols, i + 1)
        ax.imshow(img, cmap="gray")
        ax.imshow(hm, alpha=overlay_alpha)

        if show_points:
            arg = hm.reshape(-1).argmax()
            y = arg // W
            x = arg % W
            ax.scatter([x], [y], s=point_size, marker="x")

        ax.set_title(f"batch[{i}] | level_idx={level_idx}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# Model trainings

In [23]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

def training_static_aug(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    criterion,
    num_epochs=30,
    patience=5,
    use_amp=None,

    # optional schedules
    dynamic_sigma_reduction=False,
    resolution_increasing=False,
    sigma_start=7.0,
    sigma_end=3.0,
    res_start=192,
    res_end=256,

    # early-stopping margin + verbosity
    margin=1e-4,
    verbose=True,
):
    """
    Trains with optional (sigma,resolution) schedules + early stopping.
    No checkpoint saving.
    Outputs: (model_best, history) and plots train/val loss at the end.

    Assumes train_loader.dataset and val_loader.dataset implement:
      - set_sigma(float)
      - set_img_size(int)
      - have attributes: levels, img_size
    """

    if use_amp is None:
        use_amp = torch.cuda.is_available()

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_val_loss = math.inf
    best_state = None
    epochs_without_improve = 0

    train_ds_ref = train_loader.dataset
    val_ds_ref   = val_loader.dataset

    history = {
        "train_loss": [],
        "val_loss": [],
        "sigma": [],
        "img_size": [],
    }

    for epoch in range(1, num_epochs + 1):
        if verbose:
            print(f"\nEpoch {epoch}/{num_epochs}")

        # -------------------------
        # Apply schedules (optional)
        # -------------------------
        t = (epoch - 1) / max(1, (num_epochs - 1))  # 0 -> 1

        # sigma schedule
        if dynamic_sigma_reduction:
            sigma_now = sigma_start + (sigma_end - sigma_start) * t
            if hasattr(train_ds_ref, "set_sigma"):
                train_ds_ref.set_sigma(sigma_now)
            if hasattr(val_ds_ref, "set_sigma"):
                val_ds_ref.set_sigma(sigma_now)
        else:
            sigma_now = getattr(train_ds_ref, "sigma", None)

        # resolution schedule
        if resolution_increasing:
            res_now = int(round(res_start + (res_end - res_start) * t))
            if hasattr(train_ds_ref, "set_img_size"):
                train_ds_ref.set_img_size(res_now)
            if hasattr(val_ds_ref, "set_img_size"):
                val_ds_ref.set_img_size(res_now)
        else:
            res_now = getattr(train_ds_ref, "img_size", None)

        if verbose and (sigma_now is not None or res_now is not None):
            s = f"{sigma_now:.3f}" if sigma_now is not None else "None"
            print(f"  schedule -> sigma={s} | res={res_now}")

        # -------------------------
        # Train / Eval
        # -------------------------
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            criterion=criterion,
            scaler=scaler,
            use_amp=use_amp
        )

        val_loss = eval_one_epoch(
            model=model,
            loader=val_loader,
            device=device,
            criterion=criterion,
            use_amp=use_amp
        )

        history["train_loss"].append(float(train_loss))
        history["val_loss"].append(float(val_loss))
        history["sigma"].append(None if sigma_now is None else float(sigma_now))
        history["img_size"].append(None if res_now is None else int(res_now))

        if verbose:
            print(f"  Train loss: {train_loss:.6f}")
            print(f"  Val   loss: {val_loss:.6f}")

        # -------------------------
        # Early stopping / best state in RAM
        # -------------------------
        if val_loss < best_val_loss - margin:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            epochs_without_improve = 0
            if verbose:
                print("  -> New best model (kept in memory).")
        else:
            epochs_without_improve += 1
            if verbose:
                print(f"  -> No improvement ({epochs_without_improve}/{patience})")
            if epochs_without_improve >= patience:
                if verbose:
                    print("Early stopping triggered.")
                break

    # Restore best model weights (if any)
    if best_state is not None:
        model.load_state_dict(best_state)
        if verbose:
            print("\nBest model weights restored (not saved to disk).")
    else:
        if verbose:
            print("\nWarning: no best_state was recorded.")

    # -------------------------
    # Plot losses (output)
    # -------------------------
    plt.figure()
    plt.plot(history["train_loss"], label="train")
    plt.plot(history["val_loss"], label="val")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.show()

    return model, history


In [24]:
import matplotlib.pyplot as plt
import numpy as np
import torch

@torch.no_grad()
def collect_predictions_and_gt(model, loader, device):
    """
    Returns:
      gt_xy:   [N,2]
      pred_xy: [N,2]
      levels:  list of level indices (optional)
    """
    model.eval()

    gt_all = []
    pred_all = []

    for imgs, targets in loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        outputs = model(imgs)  # [B,K,H,W]

        B, K, H, W = outputs.shape

        # argmax → coordinates
        pred_flat = outputs.view(B, K, -1).argmax(dim=-1)
        gt_flat   = targets.view(B, K, -1).argmax(dim=-1)

        pred_y = (pred_flat // W).float()
        pred_x = (pred_flat %  W).float()

        gt_y = (gt_flat // W).float()
        gt_x = (gt_flat %  W).float()

        for b in range(B):
            for k in range(K):
                gt_all.append([gt_x[b,k].item(), gt_y[b,k].item()])
                pred_all.append([pred_x[b,k].item(), pred_y[b,k].item()])

    return np.array(gt_all), np.array(pred_all)





In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_gt_pred_on_ax(
    ax,
    gt_xy,
    pred_xy,
    tolerance=10,
    title="",
    draw_tolerance=False,
    max_circles=300,
    s_gt=12,
    s_pred=10,
):
    """
    Overlays GT and predicted (x,y) points in image coordinate space.
    Colors predictions by correctness within tolerance.
    """
    gt_xy = np.asarray(gt_xy)
    pred_xy = np.asarray(pred_xy)

    # correctness by euclidean distance
    dists = np.linalg.norm(pred_xy - gt_xy, axis=1)
    correct = dists <= tolerance

    # plot GT points
    ax.scatter(gt_xy[:, 0], gt_xy[:, 1], s=s_gt, marker="+", label="GT")

    # plot predictions (correct vs incorrect)
    ax.scatter(pred_xy[correct, 0], pred_xy[correct, 1], s=s_pred, alpha=0.6, label="Pred (correct)")
    ax.scatter(pred_xy[~correct, 0], pred_xy[~correct, 1], s=s_pred, alpha=0.6, label="Pred (incorrect)")

    # optionally draw tolerance circles around a subset of GT points (can be heavy if N large)
    if draw_tolerance:
        n = min(len(gt_xy), max_circles)
        idx = np.linspace(0, len(gt_xy) - 1, n).astype(int)
        for i in idx:
            circ = plt.Circle((gt_xy[i, 0], gt_xy[i, 1]), tolerance, fill=False, linewidth=0.5, alpha=0.25)
            ax.add_patch(circ)

    acc = 100.0 * correct.mean() if len(correct) else float("nan")
    ax.set_title(f"{title}\nAcc@{tolerance}px: {acc:.1f}%", fontsize=10)

    ax.set_xlabel("x (px)", fontsize=9)
    ax.set_ylabel("y (px)", fontsize=9)
    ax.grid(True, alpha=0.25)

    # image coordinate convention: origin top-left -> invert y to look like image coords
    ax.invert_yaxis()

    return correct


In [ ]:
import torch

@torch.no_grad()
def plot_val_test_gt_pred_grid(
    model,
    val_loader,
    test_loader,
    device,
    tolerance=10,
    draw_tolerance=False,
    figsize=(8.0, 3.6),
):
    gt_val, pred_val = collect_predictions_and_gt(model, val_loader, device)
    gt_test, pred_test = collect_predictions_and_gt(model, test_loader, device)

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=True, sharey=True)

    plot_gt_pred_on_ax(
        axes[0],
        gt_val,
        pred_val,
        tolerance=tolerance,
        title="Validation",
        draw_tolerance=draw_tolerance
    )

    plot_gt_pred_on_ax(
        axes[1],
        gt_test,
        pred_test,
        tolerance=tolerance,
        title="Test",
        draw_tolerance=draw_tolerance
    )

    # set limits based on combined GT range (with padding)
    all_gt = np.vstack([gt_val, gt_test])
    xmin, ymin = all_gt.min(axis=0)
    xmax, ymax = all_gt.max(axis=0)
    pad = max(10, tolerance * 2)

    for ax in axes:
        ax.set_xlim(xmin - pad, xmax + pad)
        ax.set_ylim(ymax + pad, ymin - pad)  # because y is inverted

    # one legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False, fontsize=9)

    fig.suptitle("GT vs Predicted landmark locations", fontsize=11)
    plt.tight_layout(rect=[0, 0.10, 1, 0.92])
    plt.show()


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def compare_models_pixel_error_heatmap(
    models,                 # list/tuple of 3 heatmap models
    model_names,            # list of 3 strings
    loader,                 # val_loader OR test_loader
    device,
    use_amp=True,
    levels=None,            # default: loader.dataset.levels
    figsize=(10, 5.4),
    show=True,
):
    """
    Heatmap rows: levels + ["Overall mean error", "Overall p90 error"]
    Heatmap cols: models
    Cell values: pixel error (mean per level; and overall mean / overall p90)
    Metric: Euclidean pixel distance between argmax(pred heatmap) and argmax(GT heatmap).
    Returns:
      table: np.ndarray shape [K+2, 3]
      row_names: list length K+2
    """
    assert len(models) == 3, "Pass exactly 3 models."
    assert len(model_names) == 3, "Pass exactly 3 model names."

    def argmax_xy(hm):
        # hm: [B,K,H,W] -> x,y: [B,K]
        B, K, H, W = hm.shape
        flat = hm.view(B, K, -1)
        idx = flat.argmax(dim=-1)        # [B,K]
        y = (idx // W).float()
        x = (idx %  W).float()
        return x, y

    # Infer K and level names
    imgs0, targets0 = next(iter(loader))
    K = targets0.shape[1]

    if levels is None:
        levels = getattr(loader.dataset, "levels", None)
    if levels is None:
        levels = [f"level_{i}" for i in range(K)]
    else:
        assert len(levels) == K, f"levels length ({len(levels)}) != K ({K})"

    # We'll accumulate per-level distances and global distances per model
    # Use lists of arrays to compute mean/p90 precisely without approximation.
    per_level_dists = [[[] for _ in range(K)] for _ in range(3)]
    all_dists = [[] for _ in range(3)]

    for m_idx, model in enumerate(models):
        model.eval()

        for imgs, targets in loader:
            imgs = imgs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=use_amp and torch.cuda.is_available()):
                outputs = model(imgs)  # [B,K,H,W]

            px, py = argmax_xy(outputs)
            gx, gy = argmax_xy(targets)

            dist = torch.sqrt((px - gx) ** 2 + (py - gy) ** 2)  # [B,K]
            dist_np = dist.detach().cpu().numpy()

            # store per-level and global
            for k in range(K):
                per_level_dists[m_idx][k].append(dist_np[:, k])
            all_dists[m_idx].append(dist_np.reshape(-1))

    # Build table: [K+2, 3]
    table = np.zeros((K + 2, 3), dtype=np.float64)

    for m_idx in range(3):
        # per-level mean
        for k in range(K):
            d = np.concatenate(per_level_dists[m_idx][k]) if len(per_level_dists[m_idx][k]) else np.array([])
            table[k, m_idx] = float(np.mean(d)) if d.size else np.nan

        # overall mean & p90 across all levels and samples
        d_all = np.concatenate(all_dists[m_idx]) if len(all_dists[m_idx]) else np.array([])
        table[K, m_idx]     = float(np.mean(d_all)) if d_all.size else np.nan
        table[K + 1, m_idx] = float(np.percentile(d_all, 90)) if d_all.size else np.nan

    row_names = list(levels) + ["Overall mean error", "Overall p90 error"]

    # ---- Plot heatmap with numbers ----
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(table, aspect="auto")

    ax.set_xticks(np.arange(3))
    ax.set_xticklabels(model_names, rotation=15, ha="right")

    ax.set_yticks(np.arange(K + 2))
    ax.set_yticklabels(row_names)

    ax.set_title(f"Pixel error (argmax vs GT) on {getattr(loader, 'name', 'set')} | n={len(loader.dataset)}")
    ax.set_xlabel("Model")
    ax.set_ylabel("Level / Summary")

    # annotate numbers
    for i in range(K + 2):
        for j in range(3):
            v = table[i, j]
            ax.text(j, i, "nan" if np.isnan(v) else f"{v:.2f}", ha="center", va="center", fontsize=9)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Pixel error (px)")

    plt.tight_layout()
    if show:
        plt.show()

    return table, row_names


In [26]:
import math
import torch
import matplotlib.pyplot as plt

def train_with_curriculum(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    criterion,
    num_epochs=30,
    patience=5,
    scaler=None,
    use_amp=None,
    margin=1e-4,
    verbose=True,
    plot_losses=True,
):
    """
    Training loop with:
      - train_ds.set_epoch(epoch_idx) curriculum hook
      - early stopping (best weights kept in RAM)
      - NO saving to disk
      - optional loss plot

    Expects: train_loader.dataset has:
      - set_epoch(int)
      - current_params (dict) OR None
      - levels, img_size (optional, only for info)

    Returns:
      model (restored to best_state if available), history dict
    """
    if use_amp is None:
        use_amp = torch.cuda.is_available()

    # Create scaler if not provided
    if scaler is None:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    train_ds = train_loader.dataset

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improve = 0

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, num_epochs + 1):
        # Update curriculum (0-indexed)
        if hasattr(train_ds, "set_epoch"):
            train_ds.set_epoch(epoch - 1)

        # Print curriculum params
        if verbose:
            if getattr(train_ds, "current_params", None) is not None:
                p = train_ds.current_params
                print(
                    f"\nEpoch {epoch}/{num_epochs} | curriculum: "
                    f"rot={p.get('rot_deg', float('nan')):.1f}°, "
                    f"trans={p.get('trans', float('nan')):.3f}, "
                    f"scale±{p.get('scale', float('nan')):.3f}, "
                    f"contrast±{p.get('contrast', float('nan')):.2f}, "
                    f"gamma±{p.get('gamma', float('nan')):.2f}, "
                    f"noise={p.get('noise_std', float('nan')):.3f}, "
                    f"blur_p={p.get('blur_p', float('nan')):.2f}, "
                    f"sigma={p.get('sigma', float('nan')):.2f}"
                )
            else:
                print(f"\nEpoch {epoch}/{num_epochs}")

        # Train / Eval
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            criterion=criterion,
            scaler=scaler,
            use_amp=use_amp
        )
        val_loss = eval_one_epoch(
            model=model,
            loader=val_loader,
            device=device,
            criterion=criterion,
            use_amp=use_amp
        )

        history["train_loss"].append(float(train_loss))
        history["val_loss"].append(float(val_loss))

        if verbose:
            print(f"  Train loss: {train_loss:.6f}")
            print(f"  Val   loss: {val_loss:.6f}")

        # Early stopping
        if val_loss < best_val_loss - margin:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            epochs_without_improve = 0
            if verbose:
                print("  -> New best model (kept in memory).")
        else:
            epochs_without_improve += 1
            if verbose:
                print(f"  -> No improvement ({epochs_without_improve}/{patience})")
            if epochs_without_improve >= patience:
                if verbose:
                    print("Early stopping triggered.")
                break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)
        if verbose:
            print("\nBest model weights restored (not saved).")
    else:
        if verbose:
            print("Warning: no best_state was recorded (something went wrong).")

    # Plot losses
    if plot_losses:
        plt.figure(figsize=(6.5, 4))
        plt.plot(history["train_loss"], label="train")
        plt.plot(history["val_loss"], label="val")
        plt.xlabel("epoch")
        plt.ylabel("loss")
        plt.title("Training vs Validation Loss")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    return model, history


In [ ]:
def plot_images_with_landmarks(image_classes, df_coords):
    plt.figure(figsize=(12, 12))

    for i, category in enumerate(image_classes):
        image_path = destination_dir / category
        image_in_folder = os.listdir(image_path)
        
        # Vezmeme první obrázek dané třídy
        first_image = image_in_folder[0]
        first_image_path = image_path / first_image
        
        # Načteme obrázek
        img = image.load_img(first_image_path, target_size=(IMG_SIZE, IMG_SIZE))
        img_array = image.img_to_array(img) / IMG_SIZE

        # Najdeme anotaci pro tento soubor
        matched_row = df_coords[df_coords['full_path'].str.endswith(first_image)].iloc[0]
        x_px = int(matched_row['relative_x'] * IMG_SIZE)
        y_px = int(matched_row['relative_y'] * IMG_SIZE)

        # Vykreslení
        plt.subplot(4, 4, i + 1)
        plt.imshow(img_array)
        plt.scatter(x_px, y_px, c='red', s=40)
        plt.title(f"{category}\n{first_image} ({matched_row['level']})")
        plt.axis('off')

    plt.tight_layout()
    plt.show()


In [27]:
import math
import torch
import matplotlib.pyplot as plt

def train_basic(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    criterion,
    num_epochs=30,
    patience=5,
    margin=1e-4,
    verbose=True,
    plot_losses=True,
):
    """
    Basic training loop with early stopping.
    - No saving to disk
    - Restores best weights in memory
    - Plots train/val loss

    Returns:
      model, history
    """
    best_val_loss = math.inf
    best_state = None
    epochs_without_improve = 0

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, num_epochs + 1):
        if verbose:
            print(f"\nEpoch {epoch}/{num_epochs}")

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            criterion=criterion,
        )
        val_loss = eval_one_epoch(
            model=model,
            loader=val_loader,
            device=device,
            criterion=criterion,
        )

        history["train_loss"].append(float(train_loss))
        history["val_loss"].append(float(val_loss))

        if verbose:
            print(f"  Train loss: {train_loss:.6f}")
            print(f"  Val   loss: {val_loss:.6f}")

        # Early stopping logic
        if val_loss < best_val_loss - margin:
            best_val_loss = float(val_loss)
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            epochs_without_improve = 0
            if verbose:
                print("  -> New best model (kept in memory).")
        else:
            epochs_without_improve += 1
            if verbose:
                print(f"  -> No improvement ({epochs_without_improve}/{patience})")
            if epochs_without_improve >= patience:
                if verbose:
                    print("Early stopping triggered.")
                break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)
        if verbose:
            print("\nBest model weights restored (not saved).")
    else:
        if verbose:
            print("Warning: no best_state was recorded (something went wrong).")

    # Plot losses
    if plot_losses:
        plt.figure(figsize=(6.5, 4))
        plt.plot(history["train_loss"], label="train")
        plt.plot(history["val_loss"], label="val")
        plt.xlabel("epoch")
        plt.ylabel("loss")
        plt.title("Training vs Validation Loss")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    return model, history
